Copyright 2021 The TensorFlow Authors.

You may set the runtime to use a GPU by Runtime > Change runtime type > Hardware accelerator.

In [21]:
#%pip install tf-models-official

In [22]:
#%pip uninstall -y tensorflow-privacy
#%pip install git+https://github.com/tensorflow/privacy.git


In [23]:
#%pip install dp-accounting==0.4.3

In [24]:
#%pip install tensorflow-probability

In [25]:
#%pip install tensorflow==2.14


In [26]:
#%pip install tensorflow --upgrade

In [27]:
%pip install git+https://github.com/lxuechen/private-transformers.git

  Cloning https://github.com/lxuechen/private-transformers.git to c:\users\santi\appdata\local\temp\pip-req-build-p55krjnq
  Resolved https://github.com/lxuechen/private-transformers.git to commit 18ccc4eab7355e4ac96051a82434796f6aa4624b
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/lxuechen/private-transformers.git 'C:\Users\santi\AppData\Local\Temp\pip-req-build-p55krjnq'


In [28]:
import tensorflow_privacy.privacy.privacy_tests
print(dir(tensorflow_privacy.privacy.privacy_tests))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'secret_sharer', 'version']


In [29]:
import functools
import os
import numpy as np
import tensorflow as tf
from official.utils.misc import keras_utils

from tensorflow_privacy.privacy.privacy_tests.secret_sharer.generate_secrets import SecretConfig, TextSecretProperties, generate_text_secrets_and_references, construct_secret
from tensorflow_privacy.privacy.privacy_tests.secret_sharer.exposures import compute_exposure_interpolation, compute_exposure_extrapolation

# Define functions

In [30]:
#@title Define some parameters for the model.
EMBEDDING_DIM = 256
RNN_UNITS = 1024
SEQ_LENGTH = 100

BATCHES_PER_EPOCH = 1000
TRAIN_EPOCHS = 20
BATCH_SIZE = 32
LOG_STEPS = 100

ENABLE_EAGER = True

# We will download shakespeare.txt later
TRAINING_DATA = 'shakespeare.txt'
# We will try multiple sets of secrets, so we set a placeholder for the secret's name in the directory
MODEL_DIR = 'secret{}'

In [31]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

SEQ_LENGTH = 64  # Definir la longitud de la secuencia

class SecretDataset(Dataset):
    def __init__(self, path_to_file, secrets=None, seq_length=SEQ_LENGTH):
        with open(path_to_file, 'r', encoding='utf-8') as f:
            text = f.read()

        # Crear vocabulario
        vocab = sorted(set(text))
        self.char2idx = {u: i for i, u in enumerate(vocab)}
        self.idx2char = np.array(vocab)

        # Inicializar el texto y la secuencia de entrada
        self.text = list(text)

        # Insertar los secretos en el texto
        if secrets:
            all_secrets = []
            for secret in secrets:
                secret.datasets = {}
                for r, seqs in secret.secrets.items():
                    all_secrets += seqs * r
                    indexed_seqs = np.array([[self.char2idx[c] for c in s] for s in seqs])
                    secret.datasets[r] = {
                        'data': indexed_seqs[:-1],
                        'label': indexed_seqs[1:]
                    }
                # Incluir las referencias
                indexed_seqs = np.array([[self.char2idx[c] for c in s] for s in secret.references])
                secret.datasets[0] = {
                    'data': indexed_seqs[:-1],
                    'label': indexed_seqs[1:]
                }
            
            # Insertar los secretos en el texto
            idx = np.random.choice(len(text) - 1, len(all_secrets) - 1, replace=False) + 1
            text = np.split(list(text), np.sort(idx))
            text = [''.join(t) for t in text]
            text = ''.join(np.vstack([text, all_secrets]).transpose().reshape(-1))

        # Convertir texto en índices
        self.text_as_int = np.array([self.char2idx[c] for c in self.text])

        # Dividir en secuencias de longitud seq_length + 1
        self.sequences = self._split_text_into_sequences(self.text_as_int, seq_length)

    def _split_text_into_sequences(self, text_as_int, seq_length):
        sequences = []
        for i in range(0, len(text_as_int) - seq_length, seq_length):
            sequences.append(text_as_int[i:i+seq_length+1])
        return sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = self.sequences[idx]
        input_text = torch.tensor(sequence[:-1])
        target_text = torch.tensor(sequence[1:])
        return input_text, target_text

def get_dataset(path_to_file, secrets=None, batch_size=None, seq_length=SEQ_LENGTH):
    # Crear el dataset de PyTorch
    dataset = SecretDataset(path_to_file, secrets, seq_length)

    # Crear el dataloader para hacer batching
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    return dataloader, dataset.idx2char, dataset.char2idx


In [32]:
#%pip install opacus

In [33]:

import os
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torch.optim import SGD
from transformers import get_linear_schedule_with_warmup
from private_transformers import PrivacyEngine

def collate_fn(batch):
    batch_padded = pad_sequence(batch, batch_first=True, padding_value=tokenizer.pad_token_id)
    return batch_padded.to("cpu")






def train_model(
    dataset,
    model,
    batch_size=4,
    epochs=4,
    lr=3e-1,
    warmup_steps=2500,
    device="cuda",
    output_dir=".",
    output_prefix="wreckgar",
    save_model_on_epoch=False,
    checkpoint_dir=None  # 🔹 Directorio para checkpoints
):
    model = model.to(device)
    model.train()

    optimizer = SGD(model.parameters(), lr=lr)

    privacy_engine = PrivacyEngine(
        model,
        batch_size=batch_size,
        sample_size=len(dataset),
        epochs=epochs,
        max_grad_norm=0.1,
        target_epsilon=8,
        target_delta=(1 / len(dataset)) * 2,
        clipping_mode="per_layer",
    )
    privacy_engine.attach(optimizer)

    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=-1
    )

    # Cargar último checkpoint si existe
    initial_epoch = 0
    if checkpoint_dir:
        os.makedirs(checkpoint_dir, exist_ok=True)
        checkpoint_files = sorted(
            [f for f in os.listdir(checkpoint_dir) if f.startswith("ckpt_")],
            key=lambda x: int(x.split("_")[-1].split(".")[0])
        )
        if checkpoint_files:
            last_checkpoint = os.path.join(checkpoint_dir, checkpoint_files[-1])
            checkpoint = torch.load(last_checkpoint, map_location=device)
            model.load_state_dict(checkpoint["model_state"])
            optimizer.load_state_dict(checkpoint["optimizer_state"])
            scheduler.load_state_dict(checkpoint["scheduler_state"])
            initial_epoch = checkpoint["epoch"] + 1
            print(f"✅ Loaded checkpoint from {last_checkpoint}. Resuming from epoch {initial_epoch}.")

    # Crear DataLoader con padding
    train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

    for epoch in range(initial_epoch, epochs):
        print(f"🔥 Training epoch {epoch}")

        for idx, input_tensors in enumerate(train_dataloader):
            input_tensors = input_tensors.to(device)

            # Forward pass
            outputs = model(input_tensors, labels=input_tensors)
            per_sample_loss = loss_function(outputs, input_tensors)

            optimizer.zero_grad()
            optimizer.step(loss=per_sample_loss)
            scheduler.step()

        if save_model_on_epoch and checkpoint_dir:
            checkpoint_path = os.path.join(checkpoint_dir, f"ckpt_{epoch}.pt")
            torch.save(
                {
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                },
                checkpoint_path,
            )
            print(f"📌 Checkpoint guardado en {checkpoint_path}")

    return model



In [34]:
#%pip install torch

In [35]:
#%pip install transformers

In [36]:
import os
import torch
import torch.nn.functional as F
import numpy as np
from transformers import GPT2Tokenizer, GPT2LMHeadModel

def compute_perplexity_for_secret(secrets, checkpoint_dir):
    """
    Calcula la perplejidad para cada secreto usando el checkpoint más reciente.

    Args:
        secrets: lista de secretos, cada uno con datasets[r] que contiene 'data' y 'label'
        checkpoint_dir: directorio donde se guardan los checkpoints de PyTorch

    Returns:
        ppls: diccionario con perplejidades para cada secreto en distintos datasets
        ppls_reference: perplejidad de referencia (dataset `r=0`)
    """

    # ✅ 1. Cargar el checkpoint más reciente
    checkpoints = sorted(
        [f for f in os.listdir(checkpoint_dir) if f.startswith("ckpt_")],
        key=lambda x: int(x.split("_")[-1].split(".")[0]),
    )

    if not checkpoints:
        raise ValueError(f"No se encontraron checkpoints en {checkpoint_dir}")

    latest_checkpoint = os.path.join(checkpoint_dir, checkpoints[-1])
    print(f"✅ Cargando modelo desde {latest_checkpoint}")

    model = GPT2LMHeadModel.from_pretrained("gpt2")
    model.load_state_dict(torch.load(latest_checkpoint, map_location="cuda"))
    model.to("cuda")
    model.eval()  # ⚠️ Poner en modo evaluación

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token  # Asegurar que hay un token de padding

    # ✅ 2. Evaluar perplejidad para cada secreto
    cce_loss = torch.nn.CrossEntropyLoss(reduction="none")  # Pérdida sin reducción
    ppls = [{} for _ in range(len(secrets))]
    ppls_reference = [None] * len(secrets)

    for i, secret in enumerate(secrets):
        for r, d in secret.datasets.items():
            input_texts = d['data']
            labels = d['label']

            losses = []
            for text, label in zip(input_texts, labels):
                input_ids = tokenizer.encode(text, return_tensors="pt").to("cuda")
                label_ids = tokenizer.encode(label, return_tensors="pt").to("cuda")

                with torch.no_grad():
                    outputs = model(input_ids, labels=label_ids)
                    loss = cce_loss(outputs.logits.view(-1, outputs.logits.size(-1)), label_ids.view(-1))
                    loss = loss.mean().item()  # Promediar la pérdida

                losses.append(loss)

            # ✅ 3. Convertir pérdida en perplejidad: PPL = exp(promedio de la pérdida)
            ppl = np.exp(np.mean(losses))

            if r == 0:
                ppls_reference[i] = ppl
            else:
                ppls[i][r] = ppl

    return ppls, ppls_reference


# Run training and secret sharer on Shakespeare dataset.

In [37]:
import urllib.request
url = 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
filename = 'shakespeare.txt'
print(urllib.request.urlretrieve(url, filename))

('shakespeare.txt', <http.client.HTTPMessage object at 0x000001D1020CDD30>)


In [38]:
#@title Check the vocabulary of the dataset
# (to make sure the secrets' vocabulary is a subset)
with open('shakespeare.txt', 'rb') as f:
  text = f.read().decode(encoding='utf-8')
  print(sorted(set(text)))

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [ ]:
#@title Generate secrets
# Here we present just one example and you can try other combinations of patterns,
# vocabs and number of repetitions. For example, you may use other patterns and
# different vocabularies for them.
patterns = ['{}' * 5,
            'Info ' + '{}' * 5,
            '{}{}-{}{}-{}',
            '{}' * 10]
vocabs = [list('ABCDEFGHIJ')] * 4
num_repetitions = [1, 10, 100]
num_secrets_for_repetitions = [20] * len(num_repetitions)
num_references = 65536
secret_configs = [SecretConfig(num_repetitions, num_secrets_for_repetitions, num_references,
                               properties=TextSecretProperties(vocab, pattern))
                  for vocab, pattern in zip(vocabs, patterns)]
secrets = generate_text_secrets_and_references(secret_configs)

# Let's look at the variable "secrets"
print(f'"secrets" is a list and the length is {len(secrets)} because we have four sets of secrets.')
print(f'The type of the element in "secrets" is {type(secrets[0])}.')
print(f'The field "config" contains its configuration: {secrets[0].config}')
print(f'The filed "secrets" is a dictionary, with key being the number of repetitions and value being a list of secrets.',
      f'For example, the secrets that are going to be repeated for 100 times are {secrets[0].secrets[100]}.')
print(f'The field "references" is a list of reference sequences that are not going to be injected. The first 10 elements are {secrets[0].references[:10]}.')

"secrets" is a list and the length is 4 because we have four sets of secrets.
The type of the element in "secrets" is <class 'tensorflow_privacy.privacy.privacy_tests.secret_sharer.generate_secrets.SecretsSet'>.
The field "config" contains its configuration: SecretConfig(num_repetitions=[1, 10, 100], num_secrets_for_repetitions=[20, 20, 20], num_references=65536, name='', properties=TextSecretProperties(vocab=['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J'], pattern='{}{}{}{}{}'))
The filed "secrets" is a dictionary, with key being the number of repetitions and value being a list of secrets. For example, the secrets that are going to be repeated for 100 times are ['HIEAD', 'EIAEF', 'HFEDH', 'HGCFB', 'ADBCB', 'BFDDC', 'HBDJA', 'AIJGH', 'IACCE', 'EBCHI', 'DACDJ', 'CABGG', 'FBHGD', 'GJIIB', 'AAFFB', 'EADEH', 'DICGA', 'IBBDD', 'EDICI', 'EGHHE'].
The field "references" is a list of reference sequences that are not going to be injected. The first 10 elements are ['EIHIB', 'FBJIB', 'GFAJA',

: 

In [ ]:
#@title Combine secrets with training data and train the models.
# We're going to train 4 models corresponding to the four sets of secrets,
# to examine the effect of different ways of generating secrets.
for i in range(1): #for i in range(len(secrets)):
  dataloader, idx2char, char2idx = get_dataset(
    TRAINING_DATA,
    secrets=[secrets[i]],  # Lista de secretos a insertar
    batch_size=BATCH_SIZE,
    seq_length=SEQ_LENGTH  # Añadir seq_length si es necesario
)

  print(f'Train model with secret set #{i}.')

model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
trained_model = train_model(
    dataset=dataloader, 
    model=model, 
    batch_size=BATCH_SIZE, 
    epochs=4, 
    lr=3e-1, 
    warmup_steps=2500, 
    device="cpu", 
    output_dir=".", 
    output_prefix="wreckgar", 
    save_model_on_epoch=True, 
    checkpoint_dir=MODEL_DIR.format(i)  # 📌 Guardar y cargar checkpoints aquí
)

Train model with secret set #0.


In [ ]:
#@title Secret sharer evaluation
ppls_all, ppls_reference_all = [], []
for i in range(1): #for i in range(len(secrets)):
  ppls, ppls_reference = compute_perplexity_for_secret(
      [secrets[i]], MODEL_DIR.format(i), len(idx2char))
  ppls_all += ppls
  ppls_reference_all += ppls_reference

for i, (vocab, pattern, p, p_r) in enumerate(zip(vocabs, patterns, ppls_all, ppls_reference_all)):
  exposure_interpolation = compute_exposure_interpolation(p, p_r)
  exposure_extrapolation = compute_exposure_extrapolation(p, p_r)

  # exposure_interpolation (or exposure_extrapolation) is a dictionary, where the key is the number of repetitions and
  # the value is a list of exposures (not necessarily in the same order as the secrets). We will now take a look at
  # the average exposure (with standard deviation).
  print(f'Secrets #{i+1}, vocab={vocab}, pattern={pattern}')
  print('  Interpolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_interpolation[r]):.2f}±{np.std(exposure_interpolation[r]):.2f}' for r in num_repetitions]))
  print('  Extrapolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_extrapolation[r]):.2f}±{np.std(exposure_extrapolation[r]):.2f}' for r in num_repetitions]))

We can see the following:
1. Comparing #1 vs. #2, having the prefix (#2) for the random sequences might help the model memorize.
2. Comparing #1 vs. #3, having the pattern (#3) did not help the memorization significantly. This is different from the result in the [paper](https://arxiv.org/abs/1802.08232). The effect might be model / data specific.
3. Comparing #1 vs. #4, longer secrets (#4) show higher exposure.

You can also try some other secret configurations to see if there is any difference.